# 1. Randomise abstracts for blind review

From the registration form: keep the people who want to give an oral presentation, shuffle them with a fixed seed, and write

1. an **anonymous** PDF for the reviewers (`Abstract N`, title, abstract), and
2. a **key** PDF with the same numbering plus author, career level and department, for the organisers.

The data here is synthetic (see `generate_sample_data.py`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Registration export (Google Form -> CSV). The repo ships a synthetic sample;
# point this to your own export to use it for a real event (keep that file out of git).
DATA = Path("sample_registrations.csv")
OUT = Path("output"); OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
df.columns = df.columns.str.strip()           # the form export has trailing spaces in some headers
df = df[~df["Name"].isna()]                   # drop empty rows
df = df.drop_duplicates(subset="Email address", keep="first")   # people who registered twice
print(len(df), "unique registrations")

40 unique registrations


## Select talks and shuffle

In [2]:
SEED = 2023
talks = df[df["Would you like to give an oral presentation?"] == "Yes"].copy()
talks = talks.sample(frac=1, random_state=SEED).reset_index(drop=True)   # random order
talks["Abstract number"] = np.arange(1, len(talks) + 1)
talks.to_csv(OUT / "abstracts_shuffled_with_key.csv", index=False)      # contains names -> never commit
talks[["Abstract number", "Title", "Career level"]].head()

,Abstract number,Title,Career level
0,1,Studying star-forming galaxies with integral-f...,Assistant professor
1,2,Studying cosmic voids with deep learning,Postdoc
2,3,Studying cosmic voids with a new calibration p...,Master student
3,4,Studying quasar absorption lines with deep lea...,Postdoc
4,5,Studying high-redshift galaxies with deep lear...,PhD student


## Write the PDFs

In [3]:
from xml.sax.saxutils import escape
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.platypus import PageBreak, Paragraph, SimpleDocTemplate

styles = getSampleStyleSheet()
s_title = styles["Title"]
s_body = styles["Normal"]
s_subtitle = ParagraphStyle("Subtitle", parent=s_body, fontSize=14, textColor="gray",
                            spaceAfter=10, leading=16, alignment=1)
s_identity = ParagraphStyle("Identity", parent=s_body, fontSize=14, textColor="blue",
                            spaceAfter=10, alignment=1)

def write_pdf(path, with_names):
    story = []
    for _, row in talks.iterrows():
        story.append(Paragraph(f"Abstract {row['Abstract number']}", s_title))
        if with_names:
            story.append(Paragraph(escape(f"{row['Name']} ({row['Career level']})"), s_identity))
            story.append(Paragraph(escape(row["Department"]), s_identity))
        story.append(Paragraph(escape(str(row["Title"])), s_subtitle))
        story.append(Paragraph(escape(str(row["Abstract"])), s_body))
        story.append(PageBreak())
    SimpleDocTemplate(str(path), pagesize=letter).build(story)

write_pdf(OUT / "abstracts_blind.pdf", with_names=False)
write_pdf(OUT / "abstracts_with_names.pdf", with_names=True)
print(len(talks), "abstracts written")

29 abstracts written


## Quick statistics

In [4]:
talks["Career level"].value_counts()

Career level
PhD student            8
Postdoc                7
Assistant professor    6
Master student         5
Professor              3
Name: count, dtype: int64